In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
import torch.nn.functional as F 
import os
from tqdm import tqdm
from data_loader import get_train_test_loaders, get_unlabeled_loader

In [2]:
# Hyperparameters
epochs = 5
learning_rate = 0.001
batch_size = 16
num_classes = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Load train/test data
train_loader, test_loader, _, train_set, test_set, _ = get_train_test_loaders(batch_size=batch_size, split_ratio=0.8)

# Load and modify pretrained Efficient Net B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

cuda


# Training

In [ ]:
# Training
model.train()
for epoch in range(epochs):
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=True, unit='batch')
    for i, (images, labels, path) in enumerate(progress_bar):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        avg_loss = running_loss / (i + 1)
        progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
# --- Save the Model ---
# Create the directory if it doesn't exist
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')
    print(f"Created directory: {'saved_models'}")

# Construct the full path for saving the model
full_save_path = os.path.join('saved_models', 'efficientnet_b0_model.pth')

# Save the model's state dictionary
torch.save(model.state_dict(), full_save_path)
print(f'Model state dictionary saved to: {full_save_path}')

# Evaluation

In [3]:
# Evaluation
model.load_state_dict(torch.load('saved_models/efficientnet_b0_model_5epochs.pth'))
model.eval()
correct = 0
total = 0
num_human = 0
num_ai = 0
incorrect_preds = []

with torch.no_grad():
    for images, labels, paths in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 

        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for i in range(len(labels)):
            if labels[i] == 0:
                num_human += 1
            elif labels[i] == 1:
                num_ai += 1
                
            if predicted[i] != labels[i]:
                incorrect_preds.append({
                    'path': paths[i],
                    'true_label': labels[i].item(),
                    'predicted_label': predicted[i].item(),
                    'confidence': probs[i][predicted[i]].item()
                })

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

print('Files with incorrect predictions:')
for item in incorrect_preds:
    print(f"{item['path']} -> True: {item['true_label']}, Pred: {item['predicted_label']}, Confidence: {item['confidence']:.4f}")

Test Accuracy: 99.66%
Files with incorrect predictions:
data/train_data\942e7d5aea624b45b4d476bc96c4e3c7.jpg -> True: 1, Pred: 0, Confidence: 0.5282
data/train_data\2bf7702d5fe3488f902aec1c13949794.jpg -> True: 1, Pred: 0, Confidence: 0.8500
data/train_data\ab7333cd63194d4b9006edc5d78d0698.jpg -> True: 1, Pred: 0, Confidence: 0.8236
data/train_data\f9c115a6d2d4463cb2b2ae632e7fb98a.jpg -> True: 0, Pred: 1, Confidence: 0.9102
data/train_data\2808bcfdfafc401498fe07cf9d950d3c.jpg -> True: 0, Pred: 1, Confidence: 0.9459
data/train_data\b3d067f835214f35b6db7ec8d68f44cb.jpg -> True: 1, Pred: 0, Confidence: 0.9872
data/train_data\abd486dde1734b38b118f7214b69f96a.jpg -> True: 1, Pred: 0, Confidence: 0.9666
data/train_data\70b3c78ba38d46cbacc0c24ec36791b3.jpg -> True: 1, Pred: 0, Confidence: 0.7188
data/train_data\4267e3d6170f46dcb397a8179bcde1aa.jpg -> True: 1, Pred: 0, Confidence: 0.7509
data/train_data\a2eb06e80c2b4aff9ea4373e707acfcb.jpg -> True: 0, Pred: 1, Confidence: 0.7146
data/train_dat

In [4]:
print(num_human, num_ai)

8009 7981


# Predicting unlabeled data

In [ ]:
# Unlabeled Data
unlabeled_set, _ = get_unlabeled_loader(batch_size=batch_size)

# Evaluation
model.eval()
all_probs = []
all_paths = []
with torch.no_grad():
    for images, paths in unlabeled_set:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 
        all_probs.append(probs.cpu())
        all_paths.extend(paths)


# Combine
all_probs_tensor = torch.cat(all_probs, dim=0)

# Get max probs and predicted classes
max_probs, predicted_classes = torch.max(all_probs_tensor, dim=1)

# Confidence filtering
threshold = 0.95
confident_mask = max_probs >= threshold

confident_probs = max_probs[confident_mask]
confident_preds = predicted_classes[confident_mask]
confident_paths = [all_paths[i] for i in range(len(all_paths)) if confident_mask[i]]

# Report
print(f'Found {len(confident_preds)} confident predictions out of {len(predicted_classes)}') # Maybe use this for semi-supervised learning
#for path, pred, prob in zip(confident_paths, confident_preds, confident_probs):
    #print(f'{path} -> Class {pred.item()} with probability {prob.item():.4f}')

# Making a copy of AI-generated images of test split of train

In [ ]:
import os
import shutil
# Create output directory if it doesn't exist
output_dir = "data/test_ai"
os.makedirs(output_dir, exist_ok=True)

with torch.no_grad():
    for images, labels, paths in test_loader:
            for i in range(len(labels)):
                if labels[i].item() == 1:
                    src_path = paths[i]
                    dst_path = os.path.join(output_dir, os.path.basename(src_path))
                    shutil.copy(src_path, dst_path)

# Baseline for AI-generated without augmentation - 99.07%

In [5]:
# Unlabeled Data
unlabeled_set, _ = get_unlabeled_loader(batch_size=batch_size, image_folder='data/test_ai')

# Evaluation
model.eval()
all_probs = []
all_paths = []
with torch.no_grad():
    for images, paths in unlabeled_set:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 
        all_probs.append(probs.cpu())
        all_paths.extend(paths)


# Combine
all_probs_tensor = torch.cat(all_probs, dim=0)

# Get max probs and predicted classes
max_probs, predicted_classes = torch.max(all_probs_tensor, dim=1)

# Confidence filtering
threshold = .8
confident_mask = max_probs >= threshold

confident_probs = max_probs[confident_mask]
confident_preds = predicted_classes[confident_mask]
confident_paths = [all_paths[i] for i in range(len(all_paths)) if confident_mask[i]]

# Report
print(f'Found {len(confident_preds)} confident predictions:') # Maybe use this for semi-supervised learning
#for path, pred, prob in zip(confident_paths, confident_preds, confident_probs):
    #print(f'{path} -> Class {pred.item()} with probability {prob.item():.4f}')

print(predicted_classes.sum().item()/len(predicted_classes))
print(f'{len(predicted_classes)-predicted_classes.sum().item()} images tricked the model')

Found 7879 confident predictions:
0.9907279789500063


# HSV Histogram matching - 71.98%

In [22]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from skimage.exposure import match_histograms
import cv2

augmentation = 'HSV Histogram Matching'

# Load histogram from CSV
hist_df = pd.read_csv("avg_hist_0.csv")
reference_hist = {
    'H': hist_df['H'].values,
    'S': hist_df['S'].values,
    'V': hist_df['V'].values
}

# Directories
input_dir = 'data/test_ai'
output_dir = 'data/test_ai_augmented'
os.makedirs(output_dir, exist_ok=True)

valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

# Function to match HSV histogram
def match_hsv_hist(image_rgb, reference_hist):
    image_hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    matched_hsv = np.empty_like(image_hsv)

    for i, channel in enumerate(['H', 'S', 'V']):
        src_channel = image_hsv[..., i]
        ref_hist = reference_hist[channel]

        # Generate synthetic reference array
        ref_values = np.arange(256).astype(np.uint8)
        ref_counts = (ref_hist * src_channel.size).astype(int)

        # Fix rounding mismatch
        diff = src_channel.size - ref_counts.sum()
        if diff > 0:
            ref_counts[np.argmax(ref_counts)] += diff
        elif diff < 0:
            ref_counts[np.argmax(ref_counts)] += diff

        ref_image = np.repeat(ref_values, ref_counts)
        ref_image = ref_image[:src_channel.size].reshape(src_channel.shape)

        matched_hsv[..., i] = match_histograms(src_channel, ref_image, channel_axis=None)

    matched_rgb = cv2.cvtColor(matched_hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    return matched_rgb

# Apply to all images
for filename in os.listdir(input_dir):
    if filename.lower().endswith(valid_exts):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        img = Image.open(input_path).convert('RGB')
        img_np = np.array(img)

        matched_np = match_hsv_hist(img_np, reference_hist)
        Image.fromarray(matched_np).save(output_path)

# RGB Histogram Matching - 81.52%

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from skimage.exposure import match_histograms

augmentation = 'RGB Histogram Matching'

# Load histogram from CSV
hist_df = pd.read_csv("avg_hist_0.csv")
reference_hist = {
    'R': hist_df['R'].values,
    'G': hist_df['G'].values,
    'B': hist_df['B'].values
}

# Directories
input_dir = 'data/test_ai'
output_dir = 'data/test_ai_augmented'
os.makedirs(output_dir, exist_ok=True)

valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

# Function to match RGB histogram
def match_rgb_hist(image_rgb, reference_hist):
    matched_rgb = np.empty_like(image_rgb)

    for i, channel in enumerate(['R', 'G', 'B']):
        src_channel = image_rgb[..., i]
        ref_hist = reference_hist[channel]

        # Generate synthetic reference array
        ref_values = np.arange(256).astype(np.uint8)
        ref_counts = (ref_hist * src_channel.size).astype(int)

        # Fix rounding mismatch
        diff = src_channel.size - ref_counts.sum()
        if diff > 0:
            ref_counts[np.argmax(ref_counts)] += diff
        elif diff < 0:
            ref_counts[np.argmax(ref_counts)] += diff

        ref_image = np.repeat(ref_values, ref_counts)
        ref_image = ref_image[:src_channel.size].reshape(src_channel.shape)

        matched_rgb[..., i] = match_histograms(src_channel, ref_image, channel_axis=None)

    return matched_rgb

# Apply to all images
for filename in os.listdir(input_dir):
    if filename.lower().endswith(valid_exts):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        img = Image.open(input_path).convert('RGB')
        img_np = np.array(img)

        matched_np = match_rgb_hist(img_np, reference_hist)
        Image.fromarray(matched_np).save(output_path)

# Saturation Reduction - 97.07%

In [11]:
from PIL import Image, ImageEnhance
import os

input_dir = 'data/test_ai'
output_dir = 'data/test_ai_augmented'
os.makedirs(output_dir, exist_ok=True)

augmentation = 'Saturation Reduction'

# Loop through all files in the input directory
for filename in os.listdir(input_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')):
        img_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        # Load and desaturate
        img = Image.open(img_path).convert('RGB')
        enhancer = ImageEnhance.Color(img)
        desaturated_img = enhancer.enhance(0.5)  # 50% saturation

        # Save the result
        desaturated_img.save(output_path)

print(f"Saved desaturated images to {output_dir}")

Saved desaturated images to data/test_ai_augmented


# Evaluate augmented

In [23]:
# Unlabeled Data
unlabeled_set, _ = get_unlabeled_loader(batch_size=batch_size, image_folder='data/test_ai_augmented')

# Evaluation
model.eval()
all_probs = []
all_paths = []
with torch.no_grad():
    for images, paths in unlabeled_set:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 
        all_probs.append(probs.cpu())
        all_paths.extend(paths)


# Combine
all_probs_tensor = torch.cat(all_probs, dim=0)

# Get max probs and predicted classes
max_probs, predicted_classes = torch.max(all_probs_tensor, dim=1)

# Confidence filtering
threshold = .8
confident_mask = max_probs >= threshold

confident_probs = max_probs[confident_mask]
confident_preds = predicted_classes[confident_mask]
confident_paths = [all_paths[i] for i in range(len(all_paths)) if confident_mask[i]]

# Report
print(f'Found {len(confident_preds)} confident predictions out of {len(predicted_classes)}') # Maybe use this for semi-supervised learning
#for path, pred, prob in zip(confident_paths, confident_preds, confident_probs):
    #print(f'{path} -> Class {pred.item()} with probability {prob.item():.4f}')

print(predicted_classes.sum().item()/len(predicted_classes))
print(f'{len(predicted_classes)-predicted_classes.sum().item()} images tricked the model with {augmentation} augmentation')

Found 6710 confident predictions out of 7981
0.7198346071920811
2236 images tricked the model with HSV Histogram Matching augmentation


In [ ]:
# === Target Layer for Grad-CAM ===
grads = []
features = []

def forward_hook(module, input, output):
    features.append(output)

def backward_hook(module, grad_input, grad_output):
    grads.append(grad_output[0])

# Register hooks on final conv block
target_layer = model.features[-1][0]
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

# === Load and Preprocess Image ===
def load_image(img_path):
    image = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return image, transform(image).unsqueeze(0).to(device)

# === Grad-CAM Computation ===
def compute_gradcam(input_tensor, pred_class):
    features.clear()
    grads.clear()

    output = model(input_tensor)
    model.zero_grad()
    class_score = output[0, pred_class]
    class_score.backward()

    grad = grads[0].squeeze(0)      # (C, H, W)
    feat = features[0].squeeze(0)  # (C, H, W)

    weights = grad.mean(dim=(1, 2))
    cam = torch.sum(weights[:, None, None] * feat, dim=0)
    cam = F.relu(cam)
    cam -= cam.min()
    cam /= cam.max()
    cam = cam.detach().cpu().numpy()
    cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))

    return cam

# === Overlay CAM on Original Image ===
def show_cam_on_image(image_pil, cam):
    img = np.array(image_pil.resize((cam.shape[1], cam.shape[0])))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img, 0.5, heatmap, 0.5, 0)

    plt.imshow(overlay)
    plt.title("Grad-CAM")
    plt.axis('off')
    plt.show()

# === Run Everything ===
# Replace 'your_image.jpg' with the path to your image
image_pil, input_tensor = load_image("data/test_ai_augmented/004ba51671524608ba802b702ddb0d2b.jpg")
output = model(input_tensor)
pred_class = output.argmax(dim=1).item()

cam = compute_gradcam(input_tensor, pred_class)
show_cam_on_image(image_pil, cam)


In [ ]:
from PIL import Image
from gradcam import GradCAM
from gradcam.utils.image import show_cam_on_image

# Load model
model = efficientnet_b0(pretrained=True)
model.eval()

# Load image
img = Image.open("data/test_ai_augmented/004ba51671524608ba802b702ddb0d2b.jpg").convert("RGB")
transform = Compose([
    Resize(256),
    CenterCrop(224),
    ToTensor(),
    Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
input_tensor = transform(img).unsqueeze(0)

# Target layer (final conv layer)
target_layer = model.features[-1][0]

# Run Grad-CAM
cam = GradCAM(model=model, target_layers=[target_layer], use_cuda=torch.cuda.is_available())
grayscale_cam = cam(input_tensor=input_tensor)[0]

# Convert PIL to numpy (0–1 normalized)
rgb_img = np.array(img).astype(np.float32) / 255.0

# Overlay heatmap
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
plt.imshow(visualization)
plt.axis('off')
plt.show()